In [1]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType


spark = SparkSession.builder \
    .appName("Week5_Assignment_Harsh") \
    .getOrCreate()


mock_data = [

    (1, "2023-10-01", "West", "Electronics", 150.0, None, "New York", 25, "Premium", "2023-10-01 10:00:00", "test@email.com", "user1", 150.0, 101),
    (2, "2023-10-01", "East", "Clothing", 50.0, "Active", "Boston", 35, "Basic", "2023-10-01 11:00:00", None, "user2", 50.0, 102),
    (3, "2023-10-02", "West", "Electronics", 200.0, "Active", "New York", 19, "Premium", "2023-10-02 09:30:00", "user3@email.com", "", None, 101),
    (1, "2023-10-01", "West", "Electronics", 150.0, None, "New York", 25, "Premium", "2023-10-01 10:00:00", "test@email.com", "user1", 150.0, 101),
    (4, "2023-10-03", "South", "Home", 120.0, "Inactive", "Chicago", 40, "Premium", "2023-10-03 14:15:00", "user4@email.com", "user4", 120.0, 103),
    (5, "2023-10-04", "West", "Clothing", 80.0, "Active", "New York", 22, "Premium", "2023-10-04 16:20:00", "user5@email.com", "user5", 80.0, 102)
]


columns = [
    "user_id", "transaction_date", "region", "product_category", "sale_amount",
    "status", "city", "age", "subscription", "raw_timestamp",
    "email", "username", "price", "store_id"
]


df = spark.createDataFrame(mock_data, columns)


df_sales = df


print("Original Mock Data:")
df.show()

Original Mock Data:
+-------+----------------+------+----------------+-----------+--------+--------+---+------------+-------------------+---------------+--------+-----+--------+
|user_id|transaction_date|region|product_category|sale_amount|  status|    city|age|subscription|      raw_timestamp|          email|username|price|store_id|
+-------+----------------+------+----------------+-----------+--------+--------+---+------------+-------------------+---------------+--------+-----+--------+
|      1|      2023-10-01|  West|     Electronics|      150.0|    NULL|New York| 25|     Premium|2023-10-01 10:00:00| test@email.com|   user1|150.0|     101|
|      2|      2023-10-01|  East|        Clothing|       50.0|  Active|  Boston| 35|       Basic|2023-10-01 11:00:00|           NULL|   user2| 50.0|     102|
|      3|      2023-10-02|  West|     Electronics|      200.0|  Active|New York| 19|     Premium|2023-10-02 09:30:00|user3@email.com|        | NULL|     101|
|      1|      2023-10-01|  West

In [3]:
df = df.na.fill('Unknown', subset=['status'])

In [4]:
df_cleaned = df.dropDuplicates(["user_id", "transaction_date"])

In [5]:
from pyspark.sql.functions import avg

df_result = df_sales.filter(df_sales.region == 'West') \
                    .groupBy("product_category") \
                    .agg(avg("sale_amount").alias("avg_sale_amount"))

In [6]:
from pyspark.sql.functions import col

df_city_counts = df.groupBy("city") \
                   .count() \
                   .filter(col("count") > 100)

In [7]:
df_filtered = df.filter((col("age") >= 18) &
                        (col("age") <= 30) &
                        (col("subscription") == 'Premium'))

In [8]:
df_transformed = df.withColumn("event_time", col("raw_timestamp").cast("timestamp")) \
                   .drop("raw_timestamp")

In [9]:
df_valid_users = df.filter(~(col("email").isNull() | (col("username") == "")))

In [10]:
from pyspark.sql.functions import min, max, mean

df_stats = df.agg(
    min("price").alias("min_price"),
    max("price").alias("max_price"),
    mean("price").alias("mean_price")
)

In [11]:
from pyspark.sql.functions import sum

final_pipeline_df = df.dropDuplicates() \
                      .na.fill(0, subset=["price"]) \
                      .groupBy("store_id") \
                      .agg(sum("price").alias("total_revenue"))

In [12]:
df.write.csv("data/dataset.csv", header=True, mode="overwrite")

In [13]:
final_pipeline_df.write.csv("output/results.csv", header=True, mode="overwrite")